# nb_test_unit
Unit tests for the library notebooks. Run it after every change to `nb_lib_*`.

* Each test is a function named `test_*` that uses plain `assert`.
* Tests use tiny hand-made DataFrames and a throw-away schema `zz_test`, never real tables.
* The last cell raises an error if any test fails, so it can gate a pipeline or CI.

In [ ]:
%run nb_config

In [ ]:
%run nb_lib_logging

In [ ]:
%run nb_lib_delta

## Mini test runner

In [ ]:
import io
import datetime as dt

TEST_SCHEMA = "zz_test"
spark.sql(f"DROP SCHEMA IF EXISTS {TEST_SCHEMA} CASCADE")
spark.sql(f"CREATE SCHEMA {TEST_SCHEMA}")


def rows(df, cols, order_by=None):
    """DataFrame -> sorted list of tuples, for easy comparison in asserts."""
    return [tuple(r) for r in df.select(*cols).orderBy(*(order_by or cols)).collect()]


def run_tests(namespace: dict):
    results = []
    for name in sorted(k for k, v in namespace.items() if k.startswith("test_") and callable(v)):
        try:
            namespace[name]()
            results.append((name, "PASS", ""))
        except Exception as exc:
            results.append((name, "FAIL", f"{type(exc).__name__}: {exc}"))
    for name, status, msg in results:
        print(f"{'✅' if status == 'PASS' else '❌'} {status}  {name}  {msg}")
    failed = [r for r in results if r[1] == "FAIL"]
    print(f"\n{len(results) - len(failed)} passed, {len(failed)} failed")
    return failed

## nb_config tests

In [ ]:
def test_resolve_run_date_parses_iso_string():
    assert resolve_run_date("2026-11-02") == dt.date(2026, 11, 2)


def test_resolve_run_date_empty_means_today():
    assert resolve_run_date("") == utc_now().date()
    assert resolve_run_date(None) == utc_now().date()


def test_tbl_builds_qualified_name():
    assert tbl("silver", "orders") == "silver.orders"


def test_tbl_rejects_unknown_schema():
    try:
        tbl("silverr", "orders")
    except ValueError:
        return
    raise AssertionError("expected ValueError for unknown schema")

## nb_lib_logging tests

In [ ]:
def test_logger_line_format_contains_level_name_and_run_id():
    buf = io.StringIO()
    logger = get_logger("unit", run_id="abc12345", level="INFO", stream=buf)
    logger.info("hello %s", "world")
    line = buf.getvalue().strip()
    assert "| INFO    | retail.unit | run=abc12345 | hello world" in line, line


def test_logger_has_no_duplicate_handlers_after_reinit():
    get_logger("unit2")
    logger = get_logger("unit2")
    assert len(logger.handlers) == 1


def test_run_logger_records_success_with_metrics():
    table = f"{TEST_SCHEMA}.run_log"
    with RunLogger("unit_ok", "2026-11-02", log_table=table) as run:
        run.metric("rows", 5)
    r = spark.read.table(table).filter(f"run_id = '{run.run_id}'").first()
    assert r.status == "SUCCEEDED"
    assert r.run_date == dt.date(2026, 11, 2)
    assert '"rows": 5' in r.metrics_json


def test_run_logger_records_failure_and_reraises():
    table = f"{TEST_SCHEMA}.run_log"
    run = RunLogger("unit_fail", log_table=table)
    try:
        with run:
            raise ValueError("boom")
    except ValueError:
        pass
    else:
        raise AssertionError("exception was swallowed")
    r = spark.read.table(table).filter(f"run_id = '{run.run_id}'").first()
    assert r.status == "FAILED"
    assert "boom" in r.error_message

## nb_lib_delta tests

In [ ]:
def test_dedupe_latest_keeps_newest_row_per_key():
    df = spark.createDataFrame(
        [(1, "old", "2026-11-01 10:00:00"), (1, "new", "2026-11-02 10:00:00"),
         (2, "only", "2026-11-01 09:00:00")],
        "id INT, val STRING, updated_at STRING")
    out = dedupe_latest(df, ["id"], ["updated_at"])
    assert rows(out, ["id", "val"]) == [(1, "new"), (2, "only")]


def test_dedupe_latest_collapses_exact_duplicates():
    df = spark.createDataFrame([(1, "a", "t1"), (1, "a", "t1")], "id INT, val STRING, ts STRING")
    assert dedupe_latest(df, ["id"], ["ts"]).count() == 1


def test_dedupe_latest_prefers_non_null_timestamp():
    df = spark.createDataFrame([(1, "null_ts", None), (1, "has_ts", "t1")],
                               "id INT, val STRING, ts STRING")
    assert rows(dedupe_latest(df, ["id"], ["ts"]), ["val"]) == [("has_ts",)]


def test_merge_upsert_creates_then_inserts_and_updates():
    table = f"{TEST_SCHEMA}.merge_basic"
    schema = "id INT, val STRING"
    first = merge_upsert(spark.createDataFrame([(1, "a"), (2, "b")], schema), table, ["id"])
    assert first["inserted"] == 2
    second = merge_upsert(spark.createDataFrame([(2, "B"), (3, "c")], schema), table, ["id"])
    assert second == {"inserted": 1, "updated": 1, "source_rows": 2}, second
    assert rows(spark.read.table(table), ["id", "val"]) == [(1, "a"), (2, "B"), (3, "c")]


def test_merge_upsert_update_condition_blocks_older_records():
    table = f"{TEST_SCHEMA}.merge_cond"
    schema = "id INT, status STRING, updated_at TIMESTAMP"
    newer = dt.datetime(2026, 11, 2, 12, 0)
    older = dt.datetime(2026, 11, 1, 12, 0)
    merge_upsert(spark.createDataFrame([(1, "SHIPPED", newer)], schema), table, ["id"])
    merge_upsert(spark.createDataFrame([(1, "PLACED", older)], schema), table, ["id"],
                 update_condition="s.updated_at > t.updated_at")
    assert rows(spark.read.table(table), ["id", "status"]) == [(1, "SHIPPED")]


def test_watermark_default_then_roundtrip():
    table = f"{TEST_SCHEMA}.watermark"
    spark.sql(f"CREATE TABLE {table} ({OPS_TABLES['watermark']}) USING DELTA")
    assert get_watermark("p", "orders", default="none", table=table) == "none"
    set_watermark("p", "orders", "2026-11-01 00:00:00", table=table)
    set_watermark("p", "orders", "2026-11-02 00:00:00", table=table)
    assert get_watermark("p", "orders", table=table) == "2026-11-02 00:00:00"
    assert spark.read.table(table).count() == 1

## Run

In [ ]:
failed = run_tests(globals())
spark.sql(f"DROP SCHEMA IF EXISTS {TEST_SCHEMA} CASCADE")
if failed:
    raise AssertionError(f"{len(failed)} unit test(s) failed: {[f[0] for f in failed]}")